# เทรน CNN เพิ่มจากคำตอบที่ตรวจยืนยันแล้ว

ใช้ชุดข้อมูล Private `omr_retraining_v2.zip` และเปิด GPU ถ้ามี จากนั้น Run All
ไม่ต้องนำภาพทั้งหมดขึ้น Kaggle ชุดนี้มีภาพตัดที่ตรวจสอบย้อนกลับได้ น้ำหนักเดิม และโค้ดครบแล้ว
ใช้เฉพาะ Train/Validation เลือกโมเดล และค่อยประเมิน Test เดิม 10 แผ่นหลังเลือกเสร็จ
ผลเป็น assisted development retest ไม่ใช่การทดสอบอิสระ ห้ามใช้คำตอบร่างเป็นเฉลย


In [ ]:
from pathlib import Path
import zipfile, subprocess, sys, json
archives = sorted(Path("/kaggle/input").rglob("omr_retraining_v2.zip"))
if len(archives) != 1:
    raise RuntimeError(f"Attach exactly one omr_retraining_v2.zip dataset; found {archives}")
work = Path("/kaggle/working/omr_retraining_v2")
work.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archives[0]) as z:
    z.extractall(work)
print(json.loads((work/"preflight.json").read_text())["confirmed_questions_by_split"])


In [ ]:
output = Path("/kaggle/working/retraining_results_v2")
subprocess.run([sys.executable, str(work/"src/omr_retrain.py"), "--data-dir", str(work), "--output-dir", str(output), "--epochs", "12", "--threads", "2"], check=True)


In [ ]:
evaluation = json.loads((output/"evaluation.json").read_text())
print("Selected epoch:", evaluation["selection"]["selected_epoch"])
for name in ("baseline_original_cutoff", "baseline_recalibrated_on_current_validation", "selected_model"):
    print(name, {mode: evaluation[name][mode]["accuracy"] for mode in ("A_only", "A_then_B")})
from IPython.display import FileLink, display
display(FileLink(str(output/"retraining_results.zip")))
